## Hands-on Large Language Models (O'reilly)

### Tokenization

It's used to break down the text into smaller pieces (words or part of words).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
device_map="cuda",
torch_dtype="auto",
trust_remote_code=True,
)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

There is a nice trick in trans formers that simplifies the process, namely transformers.pipeline. It encapsulates the model, tokenizer, and text generation process into a single function:

In [ ]:
from transformers import pipeline

In [ ]:
# Create a pipeline
generator = pipeline(
"text-generation",
model=model,
tokenizer=tokenizer,
return_full_text=False,
max_new_tokens=500,
do_sample=False
)

*   return_full_text: By setting this to False, the prompt will not be returned but merely the output of the model.
*   max_new_tokens: The maximum number of tokens the model will generate. By setting a limit, we prevent long and unwieldy output as some models might continue generating output until they reach their context window.
* do_sample: Whether the model uses a sampling strategy to choose the next token. By setting this to False, the model will always select the next most probable token. In Chapter 6, we explore several sampling parameters that invoke some creativity in the model’s output.



In [ ]:
# The prompt (user input / query)
messages = [
{"role": "user", "content": "Create a funny joke about chickens."}
]
# Generate output
output = generator(messages)
print(output[0]["generated_text"])

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
"microsoft/Phi-3-mini-4k-instruct",
device_map="cuda",
torch_dtype="auto",
trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap.
Explain how it happened.<|assistant|>"
# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
# Generate the text
generation_output = model.generate(
input_ids=input_ids,
max_new_tokens=20
)
# Print the output
print(tokenizer.decode(generation_output[0]))

Let’s print input_ids to see what it holds inside:

tensor([[ 1, 14350, 385, 4876, 27746, 5281, 304, 19235, 363, 278, 25305, 293,
16423, 292, 286, 728, 481, 29889, 12027, 7420, 920, 372, 9559, 29889, 32001]],
device='cuda:0')

This reveals the inputs that LLMs respond to, a series of integers as shown in
Figure 2-4. Each one is the unique ID for a specific token (character, word, or part of
a word). These IDs reference a table inside the tokenizer containing all the tokens it
knows.

In [ ]:
#This prints (each token is on a separate line):

for id in input_ids[0]:
print(tokenizer.decode(id))


* The first token is ID 1 's', a special token indicating the beginning of the text.
* Some tokens are complete words (e.g., Write, an, email).
* Some tokens are parts of words (e.g., apolog, izing, trag, ic).
* Punctuation characters are their own token.

Notice how the space character does not have its own token. Instead, partial tokens
(like “izing” and “ic”) have a special hidden character at their beginning that indicates
that they’re connected with the token that precedes them in the text. Tokens without
that special character are assumed to have a space before them.

On the output side, we can also inspect the tokens generated by the model by printing
the generation_output variable. This shows the input tokens as well as the output
tokens (we’ll highlight the new tokens in bold):

tensor([[ 1, 14350, 385, 4876, 27746, 5281, 304, 19235, 363, 278,
25305, 293, 16423, 292, 286, 728, 481, 29889, 12027, 7420,
920, 372, 9559, 29889, 32001, 3323, 622, 29901, 1619, 317,
3742, 406, 6225, 11763, 363, 278, 19906, 292, 341, 728,
481, 13, 13, 29928, 799]], device='cuda:0')

This shows us the model generated the token 3323, 'Sub', followed by token 622,
'ject'. Together they formed the word 'Subject'. They were then followed by
token 29901, which is the colon ':'...and so on. Just like on the input side, we need
the tokenizer on the output side to translate the token ID into the actual text. We do
that using the tokenizer’s decode method. We can pass it an individual token ID or a
list of them:
* print(tokenizer.decode(3323))
* print(tokenizer.decode(622))
* print(tokenizer.decode([3323, 622]))
* print(tokenizer.decode(29901))


How Does the Tokenizer Break Down Text?
There are three major factors that dictate how a tokenizer breaks down an input
prompt.

1. First, at model design time, the creator of the model chooses a tokenization method.
Popular methods include byte pair encoding (BPE) (widely used by GPT models) and
WordPiece (used by BERT). These methods are similar in that they aim to optimize
an efficient set of tokens to represent a text dataset, but they arrive at it in different
ways.

2. Second, after choosing the method, we need to make a number of tokenizer design
choices like vocabulary size and what special tokens to use.

3. Third, the tokenizer needs to be trained on a specific dataset to establish the best
vocabulary it can use to represent that dataset. Even if we set the same methods
and parameters, a tokenizer trained on an English text dataset will be different from
another trained on a code dataset or a multilingual text dataset.

#### Word Versus Subword Versus Character Versus Byte Tokens


The tokenization scheme we just discussed is called subword tokenization. It’s the most commonly used tokenization scheme but not the only one. The four notable ways to tokenize are shown in Figure 2-6. Let’s go over them:

1. **Word tokens:** This approach was common with earlier methods like word2vec but is being used less and less in NLP. Its usefulness, however, led it to be used outside of NLP for use cases such as recommendation systems, as we’ll see later in the chapter. One challenge with word tokenization is that the tokenizer may be unable to deal with new words that enter the dataset after the tokenizer was trained. This also results in a vocabulary that has a lot of tokens with minimal differences between them (e.g., apology, apologize, apologetic, apologist). This latter chal lenge is resolved by subword tokenization as it has a token for apolog, and then suffix tokens (e.g., -y, -ize, -etic, -ist) that are common with many other tokens, resulting in a more expressive vocabulary.

2. **Subwords tokens:** This method contains full and partial words. In addition to the vocabulary
expressivity mentioned earlier, another benefit of the approach is its ability to
represent new words by breaking down the new token into smaller characters,
which tend to be a part of the vocabulary. 

3. **Character tokens:** This is another method that can deal successfully with new words because it has the raw letters to fall back on. While that makes the representation easier to tokenize, it makes the modeling more difficult. Where a model with subword tokenization can represent “play” as one token, a model using character-level tokens needs to model the information to spell out “p-l-a-y” in addition to modeling the rest of the sequence. Subword tokens present an advantage over character tokens in the ability to fit more text within the limited context length of a Transformer model. So with a model with a context length of 1,024, you may be able to fit about three times as much text using subword tokenization than using character tokens (subword tokens often average three characters per token).

4. **Byte tokens:** One additional tokenization method breaks down tokens into the individual bytes that are used to represent unicode characters. Papers like “CANINE: Pre training an efficient tokenization-free encoder for language representation” out line methods like this, which are also called “tokenization-free encoding.” Other works like “ByT5: Towards a token-free future with pre-trained byte-to-byte models” show that this can be a competitive method, especially in multilingual
scenarios.

One distinction to highlight here: some subword tokenizers also include bytes as tokens in their vocabulary as the final building block to fall back to when they encounter characters they can’t otherwise represent. The GPT-2 and RoBERTa token izers do this, for example. This doesn’t make them tokenization-free byte-level token izers, because they don’t use these bytes to represent everything, only a subset,


### Comparing Trained LLM Tokenizers


In [ ]:
#different kinds of tokens:
text = """
English and CAPITALIZATION
🎵鸟
show_tokens False None elif == >= else: two tabs:" " Three tabs: "   "
12.0*50=600
"""

In [ ]:
colors_list = [
'102;194;165', '252;141;98', '141;160;203', 
'231;138;195', '166;216;84', '255;217;47'
]
def show_tokens(sentence, tokenizer_name):
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
token_ids = tokenizer(sentence).input_ids
for idx, t in enumerate(token_ids):
print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' + 
tokenizer.decode(t) + 
'\x1b[0m', 
end=' '
)

**BERT base model (uncased) (2018)**

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none
eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP]

**BERT base model (cased) (2018)**

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token
##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " "
12 . 0 * 50 = 600 [SEP]

**GPT-2 (2019)**

English 
and 
� � � � � �
CAP 
ITAL IZ 
ATION
show _ t ok 
ens 
False 
12 . 0 * 50 = 
600

**Flan-T5 (2022)** 

Tokenization method: Flan-T5 uses a tokenizer implementation called SentencePiece, introduced in “SentencePiece: A simple and language independent subword tokenizer and detokenizer for neural text processing”, which supports BPE and the unigram language model (described in “Subword regularization: Improving neural network translation models with multiple subword candidates”).

English 
and CA PI 
= else : 
two 
TAL IZ 
ATION 
<unk> 
<unk> 
tab s : " " 
Three 
tab s : " " 
show _ to 
ken s 
Fal s e 
12. 0 * 50 = 
600 
</s>
None e l if = = >

**GPT-4 (2023)**

English 
and 
CAPITAL IZATION
� � � � � �
show 
_tokens 
False 
Nonne 
elif == >= else : 
two 
tabs :"  " 
Three 
tabs : "  "
12 . 0 * 50 = 
600

**StarCoder2 (2024)**

English 
and 
CAPITAL IZATION
� � � � �
show _ 
tokens 
False 
None 
elif == >= else : 
two 
tabs :"  " 
1 2 . 0 * 5 0 = 6 0 0

**Galactica** 

English and CAP ITAL IZATION
� � � � � � �
show _ tokens False None elif == > = else : two t abs : "  " Three t abs : "  "
1 2 . 0 * 5 0 = 6 0 0

**Phi-3 and Llama 2**

English and C AP IT AL IZ ATION
� � � � � � �
show _ to kens False None elif == >= else : two tabs :"  " Three tabs : "  "
1 2 . 0 * 5 0 = 6 0 0
Tokenizer Properties
The preceding guided tour of trained tokenizers showed a number of ways in 

### Tokenizer Properties

* Tokenization methods (as we saw)
* Tokenizer parameters: vocabulary size, special tokens (common choices include: beginning of the text token, end of text token, padding token, unknown token cls token, masking token) and capitalization (do we want to deal with capitalization? should we convert everythin to lowercase?)